# Particle filter regime allocation

A particle filter estimates market regimes from equity and volatility-index features and changes the allocation across growth and defensive assets.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
PARTICLE FILTER STRATEGY - COMPLETE IMPLEMENTATION
===================================================

CRITICAL FIX: Proper column handling during VIX merge
- VIX data reduced to only necessary columns before merge
- Prevents pandas from creating _x/_y column suffixes
- Full backtest engine included
- Comprehensive visualizations

DATA SOURCES:
- S&P 500: Yahoo Finance (^GSPC)
- VIX: Yahoo Finance (^VIX)
- VIX9D: yfinance (not available on FMP)
- All assets: Yahoo Finance

EXECUTION FLOW:
Day T-1 (Close):
├─ Portfolio valued at Close[T-1]
├─ Features from T-2 data (double-lagged)
├─ Regime detected using particle filter
├─ Optimization using data up to T-1
├─ Signals generated
└─ Sizing info stored (uses Close[T-1] value)

Day T (Open):
├─ Execute pending signals at Open[T]
├─ Use Close[T-1] sizing (no look-ahead)
└─ Portfolio positions updated

Day T (Close):
├─ Portfolio valued at Close[T]
├─ Returns: Close[T]/Close[T-1] - 1
└─ This value becomes Close[T-1] for T+1 sizing
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
import networkx as nx
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# PORTFOLIO OPTIMIZERS
# ============================================================================

class MaxSharpeOptimizer:
    """Maximum Sharpe Ratio Optimizer - Long Only with Leverage"""
    
    def __init__(self, risk_free_rate: float = 0.02, max_exposure: float = 1.2):
        self.risk_free_rate = risk_free_rate
        self.max_exposure = max_exposure
    
    def optimize(self, returns: pd.DataFrame, max_weight: float = 0.15) -> Dict[str, float]:
        """Optimize for maximum Sharpe ratio - LONG ONLY"""
        if len(returns) < 60:
            return {}
        
        expected_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        
        try:
            lw = LedoitWolf()
            lw.fit(returns.values)
            if hasattr(lw, 'covariance_'):
                cov_shrunk = lw.covariance_
                cov_matrix = pd.DataFrame(cov_shrunk * 252,
                                         index=returns.columns,
                                         columns=returns.columns)
        except:
            pass
        
        n_assets = len(expected_returns)
        
        def neg_sharpe(weights):
            port_return = np.dot(weights, expected_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            if port_vol < 1e-6 or not np.isfinite(port_vol):
                return 1e6
            sharpe = (port_return - self.risk_free_rate) / port_vol
            if not np.isfinite(sharpe):
                return 1e6
            return -sharpe
        
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - self.max_exposure}]
        bounds = [(0, max_weight) for _ in range(n_assets)]
        x0 = np.array([self.max_exposure / n_assets] * n_assets)
        
        try:
            result = minimize(neg_sharpe, x0, method='SLSQP',
                            bounds=bounds, constraints=constraints,
                            options={'maxiter': 1000, 'ftol': 1e-9})
            
            if result.success:
                weights = result.x
                weights = np.where(weights < 0.01, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * self.max_exposure / np.sum(weights)
                    return {asset: w for asset, w in zip(expected_returns.index, weights) if w > 0.01}
        except:
            pass
        
        return {}


class HRPOptimizer:
    """Hierarchical Risk Parity - 80% Exposure"""
    
    def __init__(self, linkage_method: str = 'ward', target_exposure: float = 0.8):
        self.linkage_method = linkage_method
        self.target_exposure = target_exposure
    
    def optimize(self, returns: pd.DataFrame) -> Dict[str, float]:
        if len(returns) < 60:
            return {}
        
        try:
            corr_matrix = returns.corr().fillna(0)
            if not np.isfinite(corr_matrix.values).all():
                return {}
            
            distance_matrix = np.sqrt(0.5 * (1 - corr_matrix))
            condensed_distances = squareform(distance_matrix.values)
            linkage_matrix = linkage(condensed_distances, method=self.linkage_method)
            
            sorted_indices = self._get_quasi_diagonal(linkage_matrix, len(corr_matrix))
            sorted_corr = corr_matrix.iloc[sorted_indices, sorted_indices]
            hrp_weights = self._get_recursive_bisection(sorted_corr)
            
            weight_dict = {}
            for i, asset in enumerate(corr_matrix.index):
                original_index = sorted_indices.index(i)
                weight = hrp_weights[original_index] * self.target_exposure
                if weight > 0.01:
                    weight_dict[asset] = weight
            
            return weight_dict
        except:
            return {}
    
    def _get_quasi_diagonal(self, linkage_matrix, num_assets):
        sorted_indices = []
        def append_order(node_index):
            if node_index < num_assets:
                sorted_indices.append(node_index)
            else:
                left = int(linkage_matrix[node_index - num_assets][0])
                right = int(linkage_matrix[node_index - num_assets][1])
                append_order(left)
                append_order(right)
        append_order(len(linkage_matrix) + num_assets - 1)
        return sorted_indices
    
    def _get_recursive_bisection(self, corr_matrix):
        weights = pd.Series(1.0, index=corr_matrix.index)
        clustered_alphas = [corr_matrix.index.tolist()]
        
        while len(clustered_alphas) > 0:
            clustered_alphas = [cluster[start:end]
                              for cluster in clustered_alphas
                              for start, end in ((0, len(cluster)//2),
                                                (len(cluster)//2, len(cluster)))
                              if len(cluster) > 1]
            
            for i in range(0, len(clustered_alphas), 2):
                if i + 1 < len(clustered_alphas):
                    left = clustered_alphas[i]
                    right = clustered_alphas[i + 1]
                    
                    left_var = self._get_cluster_var(corr_matrix, left)
                    right_var = self._get_cluster_var(corr_matrix, right)
                    
                    total_var = left_var + right_var
                    if total_var > 0:
                        left_weight = right_var / total_var
                        right_weight = left_var / total_var
                        
                        for asset in left:
                            weights[asset] *= left_weight
                        for asset in right:
                            weights[asset] *= right_weight
        
        return weights.values
    
    def _get_cluster_var(self, corr_matrix, cluster):
        if len(cluster) == 1:
            return 1.0
        cluster_corr = corr_matrix.loc[cluster, cluster]
        weights = np.ones(len(cluster)) / len(cluster)
        return np.dot(weights.T, np.dot(cluster_corr.values, weights))


class MSTOptimizer:
    """Minimum Spanning Tree - 25% Exposure"""
    
    def __init__(self, diversification_factor: float = 0.8, target_exposure: float = 0.25):
        self.diversification_factor = diversification_factor
        self.target_exposure = target_exposure
    
    def optimize(self, returns: pd.DataFrame) -> Dict[str, float]:
        if len(returns) < 60:
            return {}
        
        try:
            corr_matrix = returns.corr().fillna(0)
            distance_matrix = np.sqrt(2 * (1 - corr_matrix))
            np.fill_diagonal(distance_matrix.values, 0)
            
            G = nx.Graph()
            assets = list(corr_matrix.index)
            
            for asset in assets:
                G.add_node(asset)
            
            for i, asset1 in enumerate(assets):
                for j, asset2 in enumerate(assets[i+1:], i+1):
                    distance = distance_matrix.iloc[i, j]
                    if not np.isnan(distance) and distance > 0:
                        G.add_edge(asset1, asset2, weight=distance)
            
            mst = nx.minimum_spanning_tree(G)
            centrality = nx.degree_centrality(mst)
            volatilities = returns.std()
            
            combined_scores = {}
            for asset in assets:
                centrality_score = centrality.get(asset, 0)
                vol_score = 1.0 / (volatilities[asset] + 1e-6) if asset in volatilities.index else 1.0
                combined_scores[asset] = (self.diversification_factor * centrality_score +
                                        (1 - self.diversification_factor) * vol_score)
            
            total_score = sum(combined_scores.values())
            if total_score > 0:
                weight_dict = {}
                for asset, score in combined_scores.items():
                    weight = (score / total_score) * self.target_exposure
                    if weight > 0.01:
                        weight_dict[asset] = weight
                return weight_dict
        except:
            pass
        
        return {}


# ============================================================================
# DATA FETCHER
# ============================================================================

class MarketDataFetcher:
    """Data fetcher with column validation"""
    
    
    def fetch_historical_prices(self, symbol: str, start_date: str, end_date: str, buffer_days: int = 150) -> Optional[pd.DataFrame]:
        buffer_start = pd.Timestamp(start_date) - pd.Timedelta(days=buffer_days)
        try:
            frame = download_ohlcv(symbol.replace("%5E", "^"), buffer_start, end_date)
        except RuntimeError:
            return None
        if len(frame) < 50:
            return None
        return frame.rename_axis("Date").reset_index()[["Date", "Open", "Close", "Volume"]]
    
    def fetch_vix9d_yfinance(self, start_date: str, end_date: str, buffer_days: int = 150) -> Optional[pd.DataFrame]:
        buffer_start = pd.Timestamp(start_date) - pd.Timedelta(days=buffer_days)
        try:
            frame = download_ohlcv("^VIX9D", buffer_start, end_date, end_inclusive=False)
        except RuntimeError:
            return None
        if len(frame) < 50:
            return None
        return frame.rename_axis("Date").reset_index()[["Date", "Close"]]
    
    def fetch_multiple_assets(self, symbols: List[str], start_date: str,
                             end_date: str, min_success_rate: float = 0.75) -> Dict[str, pd.DataFrame]:
        """Fetch multiple assets"""
        asset_data = {}
        successful = 0
        
        for symbol in symbols:
            df = self.fetch_historical_prices(symbol, start_date, end_date)
            if df is not None:
                asset_data[symbol] = df
                successful += 1
        
        success_rate = successful / len(symbols)
        if success_rate < min_success_rate:
            raise ValueError(f"Insufficient coverage: {successful}/{len(symbols)} ({success_rate:.1%})")
        
        return asset_data


# ============================================================================
# PARTICLE FILTER MODEL
# ============================================================================

class SP500RegimeFilter:
    """Particle Filter for S&P 500 Regime Detection"""
    
    def __init__(
        self,
        n_particles: int=1000,
        initial_capital: float=100000.0,
        rebalance_frequency: int=21,
        benchmark_symbol: str='URTH',
        risk_free_rate: float=0.02,
        commission_rate: float=0.0005,
        bid_ask_spread: float=0.0003,
        market_impact: float=0.0002,
        min_commission: float=1.0,
        max_position_size: float=0.15,
        min_optimization_history: int=126,
        cdar_threshold: float=-0.05,
        upside_threshold: float=0.1,
        short_allocation: float=0.15,
        min_asset_success_rate: float=0.75,
        min_data_points_for_optimization: int=60,
        trade_size_floor_pct: float=0.005,
        max_missing_data_pct: float=0.2,
        verbose: bool=True,
    ):
        
        self.n_particles = n_particles
        self.states = ['Bearish', 'Sideways', 'Bullish']
        self.n_states = len(self.states)
        self.verbose = verbose
        
        self.initial_capital = initial_capital
        self.rebalance_frequency = rebalance_frequency
        self.benchmark_symbol = benchmark_symbol
        self.risk_free_rate = risk_free_rate
        self.daily_rf_rate = risk_free_rate / 252
        
        self.commission_rate = commission_rate
        self.bid_ask_spread = bid_ask_spread
        self.market_impact = market_impact
        self.min_commission = min_commission
        
        self.max_position_size = max_position_size
        self.min_optimization_history = min_optimization_history
        
        self.cdar_threshold = cdar_threshold
        self.upside_threshold = upside_threshold
        self.short_allocation = short_allocation
        
        self.min_asset_success_rate = min_asset_success_rate
        self.min_data_points_for_optimization = min_data_points_for_optimization
        self.trade_size_floor_pct = trade_size_floor_pct
        self.max_missing_data_pct = max_missing_data_pct
        
        self.particles = np.random.dirichlet([1, 1, 1], n_particles)
        self.weights = np.ones(n_particles) / n_particles
        self.current_regime_probs = np.zeros(self.n_states)
        
        self.regime_params = {
            'cc_mean_returns': np.array([-0.001, 0.0003, 0.0008]),
            'cc_volatilities': np.array([0.025, 0.012, 0.015]),
            'oo_mean_returns': np.array([-0.0008, 0.0002, 0.0006]),
            'oo_volatilities': np.array([0.030, 0.018, 0.022]),
            'co_mean_returns': np.array([-0.0003, 0.0001, 0.0002]),
            'co_volatilities': np.array([0.015, 0.008, 0.012]),
            'vix_means': np.array([28.0, 18.0, 15.0]),
            'vix_stds': np.array([8.0, 4.0, 3.0])
        }
        
        self.transition_matrix = np.array([
            [0.85, 0.10, 0.05],
            [0.15, 0.70, 0.15],
            [0.05, 0.10, 0.85]
        ])
        
        self.asset_symbols = [
            'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA',
            'XLF', 'XLE', 'XLK', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU',
            'EFA', 'EWJ', 'EWG', 'EWC', 'EEM', 'VWO',
            'TLT', 'IEF', 'LQD', 'HYG', 'GLD', 'SLV', 'VNQ'
        ]
        
        self.short_symbol = 'SPY'
        
        self.max_sharpe_optimizer = MaxSharpeOptimizer(risk_free_rate, max_exposure=1.2)
        self.hrp_optimizer = HRPOptimizer(target_exposure=0.8)
        self.mst_optimizer = MSTOptimizer(target_exposure=0.25)
        
        self.data_fetcher = MarketDataFetcher()
        
        self.asset_data = {}
        self.backtest_results = None
        
        if self.verbose:
            print("="*80)
            print("PARTICLE FILTER - REGIME DETECTION STRATEGY")
            print("="*80)
    
    def calculate_features(self, sp500_df: pd.DataFrame, 
                          vix_df: pd.DataFrame) -> Tuple[pd.DataFrame, bool]:
        """Calculate features with double-lagging (T-2 for T signals)"""
        df = sp500_df.copy()
        
        # CRITICAL FIX: Only merge necessary VIX columns
        vix_cols_to_merge = ['Date', 'VIX']
        if 'VIX9D' in vix_df.columns:
            vix_cols_to_merge.append('VIX9D')
        
        df = df.merge(vix_df[vix_cols_to_merge], on='Date', how='left')
        
        df['VIX'] = df['VIX'].fillna(method='ffill', limit=5)
        
        if 'VIX9D' in df.columns:
            df['VIX9D'] = df['VIX9D'].fillna(method='ffill', limit=5)
            use_vix9d = True
        else:
            use_vix9d = False
        
        df = df.dropna(subset=['VIX'])
        
        # Calculate returns
        df['close_to_close'] = df['Close'].pct_change()
        df['open_to_open'] = df['Open'].pct_change()
        df['close_to_open'] = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)
        
        # FIRST LAG
        df['cc_T_minus_1'] = df['close_to_close'].shift(1)
        df['oo_T_minus_1'] = df['open_to_open'].shift(1)
        df['co_T_minus_1'] = df['close_to_open'].shift(1)
        
        df['rolling_vol_21_T_minus_1'] = df['cc_T_minus_1'].rolling(
            window=21, min_periods=21
        ).std() * np.sqrt(252)
        
        df['VIX_T_minus_1'] = df['VIX'].shift(1)
        if use_vix9d:
            df['VIX9D_T_minus_1'] = df['VIX9D'].shift(1)
        
        # SECOND LAG
        df['cc_T_minus_1_for_signal'] = df['cc_T_minus_1'].shift(1)
        df['oo_T_minus_1_for_signal'] = df['oo_T_minus_1'].shift(1)
        df['co_T_minus_1_for_signal'] = df['co_T_minus_1'].shift(1)
        df['rolling_vol_21_for_signal'] = df['rolling_vol_21_T_minus_1'].shift(1)
        df['VIX_for_signal'] = df['VIX_T_minus_1'].shift(1)
        if use_vix9d:
            df['VIX9D_for_signal'] = df['VIX9D_T_minus_1'].shift(1)
        
        df['close_to_close_current'] = df['close_to_close']
        
        df = df.dropna()
        
        return df, use_vix9d
    
    def calculate_likelihood(self, features: Dict, regime: int, use_vix9d: bool) -> float:
        """Calculate likelihood"""
        try:
            cc_return = features.get('cc_T_minus_1_for_signal', 0.0)
            oo_return = features.get('oo_T_minus_1_for_signal', 0.0)
            co_return = features.get('co_T_minus_1_for_signal', 0.0)
            vix_level = features.get('VIX_for_signal', 20.0)
            vol_level = features.get('rolling_vol_21_for_signal', 0.15)
            
            if use_vix9d:
                vix9d_level = features.get('VIX9D_for_signal', 18.0)
            
            checks = [cc_return, oo_return, co_return, vix_level, vol_level]
            if use_vix9d:
                checks.append(vix9d_level)
            
            if any(pd.isna(checks)):
                return 1e-10
            
            cc_mean = self.regime_params['cc_mean_returns'][regime]
            cc_vol = self.regime_params['cc_volatilities'][regime]
            oo_mean = self.regime_params['oo_mean_returns'][regime]
            oo_vol = self.regime_params['oo_volatilities'][regime]
            co_mean = self.regime_params['co_mean_returns'][regime]
            co_vol = self.regime_params['co_volatilities'][regime]
            vix_mean = self.regime_params['vix_means'][regime]
            vix_std = self.regime_params['vix_stds'][regime]
            
            ll_cc = -0.5 * ((cc_return - cc_mean) / (cc_vol + 1e-8)) ** 2
            ll_oo = -0.5 * ((oo_return - oo_mean) / (oo_vol + 1e-8)) ** 2
            ll_co = -0.5 * ((co_return - co_mean) / (co_vol + 1e-8)) ** 2
            ll_vix = -0.5 * ((vix_level - vix_mean) / (vix_std + 1e-8)) ** 2
            
            if use_vix9d:
                ll_vix9d = -0.5 * ((vix9d_level - vix_mean * 0.9) / (vix_std * 0.8 + 1e-8)) ** 2
            else:
                ll_vix9d = 0
            
            expected_vol = cc_vol * np.sqrt(252)
            ll_vol = -0.5 * ((vol_level - expected_vol) / (expected_vol * 0.3 + 1e-8)) ** 2
            
            if use_vix9d:
                total_ll = (ll_cc * 1.0 + ll_oo * 1.0 + ll_co * 0.8 +
                           ll_vix * 0.9 + ll_vix9d * 0.7 + ll_vol * 0.6)
            else:
                total_ll = (ll_cc * 1.0 + ll_oo * 1.0 + ll_co * 0.8 +
                           ll_vix * 1.2 + ll_vol * 0.8)
            
            total_ll = np.clip(total_ll, -20, 5)
            likelihood = np.exp(total_ll)
            
            if likelihood < 1e-15 or not np.isfinite(likelihood):
                likelihood = 1e-15
            
            return likelihood
        except:
            return 1e-15
    
    def update_particles(self, features: Dict, use_vix9d: bool):
        """Update particle weights"""
        particle_weights = np.zeros(self.n_particles)
        
        for i in range(self.n_particles):
            total_weight = 0
            for regime in range(self.n_states):
                regime_prob = self.particles[i, regime]
                likelihood = self.calculate_likelihood(features, regime, use_vix9d)
                total_weight += regime_prob * likelihood
            particle_weights[i] = total_weight
        
        particle_weights = np.clip(particle_weights, 1e-15, np.inf)
        
        weight_sum = np.sum(particle_weights)
        if weight_sum > 0:
            self.weights = particle_weights / weight_sum
        else:
            self.weights = np.ones(self.n_particles) / self.n_particles
        
        eff_sample_size = 1.0 / np.sum(self.weights ** 2)
        if eff_sample_size < max(self.n_particles / 3, 50):
            self.resample()
    
    def resample(self):
        """Resample particles"""
        indices = np.random.choice(self.n_particles, size=self.n_particles,
                                  p=self.weights, replace=True)
        self.particles = self.particles[indices].copy()
        
        for i in range(self.n_particles):
            noise = np.random.normal(0, 0.02, self.n_states)
            self.particles[i] += noise
            self.particles[i] = np.abs(self.particles[i])
            
            particle_sum = np.sum(self.particles[i])
            if particle_sum > 0:
                self.particles[i] = self.particles[i] / particle_sum
            else:
                self.particles[i] = np.ones(self.n_states) / self.n_states
        
        self.weights = np.ones(self.n_particles) / self.n_particles
    
    def predict(self, features: Dict, use_vix9d: bool) -> Tuple[np.ndarray, int]:
        """Predict regime"""
        for i in range(self.n_particles):
            try:
                current_regime = np.argmax(self.particles[i])
                transition_probs = self.transition_matrix[current_regime]
                transition_probs = transition_probs * 0.95 + 0.05 / self.n_states
                self.particles[i] = transition_probs
            except:
                self.particles[i] = np.ones(self.n_states) / self.n_states
        
        self.update_particles(features, use_vix9d)
        
        try:
            self.current_regime_probs = np.average(self.particles, weights=self.weights, axis=0)
        except:
            self.current_regime_probs = np.mean(self.particles, axis=0)
        
        if not np.isfinite(self.current_regime_probs).all():
            self.current_regime_probs = np.ones(self.n_states) / self.n_states
        
        prob_sum = np.sum(self.current_regime_probs)
        if prob_sum > 0:
            self.current_regime_probs = self.current_regime_probs / prob_sum
        else:
            self.current_regime_probs = np.ones(self.n_states) / self.n_states
        
        most_likely_regime = np.argmax(self.current_regime_probs)
        
        return self.current_regime_probs, most_likely_regime
    
    def get_eligible_assets_at_date(self, date: pd.Timestamp,
                                   min_history: int = 63) -> List[str]:
        """Get eligible assets (no survivorship bias)"""
        eligible = []
        
        for asset in self.asset_symbols:
            if asset not in self.asset_data:
                continue
            
            asset_df = self.asset_data[asset]
            historical_data = asset_df[asset_df['Date'] <= date]
            
            if len(historical_data) >= min_history:
                latest_date = historical_data['Date'].max()
                days_ago = (date - latest_date).days
                
                if days_ago <= 5:
                    eligible.append(asset)
        
        return eligible
    
    def build_returns_matrix_at_date(self, date: pd.Timestamp,
                                    eligible_assets: List[str],
                                    lookback: int = 252) -> pd.DataFrame:
        """Build returns matrix"""
        end_date = date
        start_date = date - timedelta(days=lookback * 2)
        
        returns_dict = {}
        
        for asset in eligible_assets:
            if asset not in self.asset_data:
                continue
            
            asset_df = self.asset_data[asset]
            historical = asset_df[
                (asset_df['Date'] >= start_date) &
                (asset_df['Date'] <= end_date)
            ].copy()
            
            if len(historical) < lookback // 2:
                continue
            
            historical['return'] = historical['Close'].pct_change()
            returns_series = historical.set_index('Date')['return']
            returns_dict[asset] = returns_series
        
        if not returns_dict:
            return pd.DataFrame()
        
        returns_matrix = pd.DataFrame(returns_dict)
        
        min_assets = max(5, len(returns_matrix.columns) * (1 - self.max_missing_data_pct))
        returns_matrix = returns_matrix.dropna(thresh=min_assets)
        
        return returns_matrix
    
    def optimize_portfolio(self, regime: int, date: pd.Timestamp,
                          eligible_assets: List[str]) -> Dict[str, float]:
        """Optimize portfolio"""
        if len(eligible_assets) < 5:
            return {}
        
        returns_matrix = self.build_returns_matrix_at_date(
            date, eligible_assets, self.min_optimization_history * 2
        )
        
        if returns_matrix.empty or len(returns_matrix) < self.min_data_points_for_optimization:
            return {}
        
        recent_returns = returns_matrix.tail(self.min_optimization_history)
        
        if len(recent_returns) < self.min_data_points_for_optimization:
            return {}
        
        try:
            if regime == 0:
                weights = self.mst_optimizer.optimize(recent_returns)
            elif regime == 1:
                weights = self.hrp_optimizer.optimize(recent_returns)
            elif regime == 2:
                weights = self.max_sharpe_optimizer.optimize(recent_returns, self.max_position_size)
            else:
                return {}
            
            return weights
        except:
            return {}
    
    def calculate_trade_costs(self, trade_value: float) -> float:
        """Calculate transaction costs"""
        abs_value = abs(trade_value)
        commission = max(self.commission_rate * abs_value, self.min_commission)
        spread = abs_value * self.bid_ask_spread
        impact = abs_value * self.market_impact
        return commission + spread + impact
    
    def run_backtest(self, start_date: str, end_date: str) -> Dict:
        """
        Run complete backtest with particle filter regime detection.
        
        EXECUTION FLOW (NO LOOK-AHEAD):
        --------------------------------
        Day T-1 Close:
          - Observe Close[T-1]
          - Calculate features using T-2 data
          - Detect regime using particle filter
          - Optimize portfolio using data up to T-1
          - Generate signals
          - Calculate target positions using Close[T-1] valuation
        
        Day T Open:
          - Execute at Open[T]
          - Apply transaction costs
          - Update positions
        
        Day T Close:
          - Value portfolio at Close[T]
          - Calculate returns: Close[T]/Close[T-1] - 1
        """
        if self.verbose:
            print(f"\n{'='*80}")
            print(f"BACKTEST: {start_date} to {end_date}")
            print(f"{'='*80}")
        
        # ================================================================
        # PHASE 1: DATA ACQUISITION
        # ================================================================
        if self.verbose:
            print(f"\nPHASE 1: DATA ACQUISITION")
            print("-"*80)
        
        sp500_data = self.data_fetcher.fetch_historical_prices(
            '^GSPC', start_date, end_date, buffer_days=150
        )
        if sp500_data is None:
            raise ValueError("Failed to fetch S&P 500")
        
        vix_data = self.data_fetcher.fetch_historical_prices(
            '^VIX', start_date, end_date, buffer_days=150
        )
        if vix_data is None:
            raise ValueError("Failed to fetch VIX")
        
        vix9d_data = self.data_fetcher.fetch_vix9d_yfinance(start_date, end_date, buffer_days=150)
        
        # CRITICAL FIX: Rename columns to avoid conflicts
        vix_combined = vix_data[['Date', 'Close']].rename(columns={'Close': 'VIX'})
        if vix9d_data is not None:
            vix_combined = vix_combined.merge(
                vix9d_data.rename(columns={'Close': 'VIX9D'}),
                on='Date', how='left'
            )
        
        # Fetch assets
        all_symbols = list(set(self.asset_symbols + [self.benchmark_symbol, self.short_symbol]))
        self.asset_data = self.data_fetcher.fetch_multiple_assets(
            all_symbols, start_date, end_date, self.min_asset_success_rate
        )
        
        if self.verbose:
            print(f"✓ Fetched {len(self.asset_data)} assets")
        
        # ================================================================
        # PHASE 2: FEATURE CALCULATION
        # ================================================================
        if self.verbose:
            print(f"\nPHASE 2: FEATURE CALCULATION")
            print("-"*80)
        
        features_df, use_vix9d = self.calculate_features(sp500_data, vix_combined)
        
        if self.verbose:
            print(f"✓ Features: {len(features_df)} observations")
        
        # ================================================================
        # PHASE 3: BACKTEST EXECUTION
        # ================================================================
        if self.verbose:
            print(f"\nPHASE 3: BACKTEST EXECUTION")
            print("-"*80)
        
        # Initialize tracking
        portfolio_history = []
        regime_history = []
        trade_history = []
        
        # Portfolio state
        capital = self.initial_capital
        cash = self.initial_capital
        positions = {}
        short_position = 0.0
        
        last_rebalance_idx = -self.rebalance_frequency
        
        # Backtest loop
        for idx, row in features_df.iterrows():
            current_date = row['Date']
            
            # Progress indicator
            if self.verbose and idx % 200 == 0:
                progress = idx / len(features_df) * 100
                print(f"  Progress: {progress:.1f}% ({idx}/{len(features_df)})")
            
            # ========================================================
            # STEP 1: REGIME DETECTION (using T-2 features)
            # ========================================================
            feature_dict = {
                'cc_T_minus_1_for_signal': row['cc_T_minus_1_for_signal'],
                'oo_T_minus_1_for_signal': row['oo_T_minus_1_for_signal'],
                'co_T_minus_1_for_signal': row['co_T_minus_1_for_signal'],
                'rolling_vol_21_for_signal': row['rolling_vol_21_for_signal'],
                'VIX_for_signal': row['VIX_for_signal']
            }
            
            if use_vix9d:
                feature_dict['VIX9D_for_signal'] = row['VIX9D_for_signal']
            
            regime_probs, regime = self.predict(feature_dict, use_vix9d)
            
            # Store regime
            regime_history.append({
                'date': current_date,
                'regime': self.states[regime],
                'regime_idx': regime,
                'bearish_prob': regime_probs[0],
                'sideways_prob': regime_probs[1],
                'bullish_prob': regime_probs[2]
            })
            
            # ========================================================
            # STEP 2: PORTFOLIO VALUATION (at Close[T])
            # ========================================================
            # Value all positions at Close[T]
            portfolio_value_long = 0.0
            for asset, shares in positions.items():
                if asset in self.asset_data:
                    asset_df = self.asset_data[asset]
                    price_row = asset_df[asset_df['Date'] == current_date]
                    
                    if not price_row.empty:
                        close_price = price_row.iloc[0]['Close']
                        portfolio_value_long += shares * close_price
            
            # Value short position
            portfolio_value_short = 0.0
            if short_position != 0 and self.short_symbol in self.asset_data:
                short_df = self.asset_data[self.short_symbol]
                short_price_row = short_df[short_df['Date'] == current_date]
                
                if not short_price_row.empty:
                    short_close = short_price_row.iloc[0]['Close']
                    portfolio_value_short = short_position * short_close
            
            # Cash earns risk-free rate
            if idx > 0:
                cash += cash * self.daily_rf_rate
            
            # Total portfolio value
            portfolio_value = cash + portfolio_value_long + portfolio_value_short
            
            # Store portfolio state
            portfolio_history.append({
                'date': current_date,
                'portfolio_value': portfolio_value,
                'cash': cash,
                'long_value': portfolio_value_long,
                'short_value': portfolio_value_short,
                'regime': self.states[regime],
                'num_positions': len([p for p in positions.values() if p != 0])
            })
            
            # ========================================================
            # STEP 3: REBALANCING LOGIC
            # ========================================================
            days_since_rebal = idx - last_rebalance_idx
            should_rebalance = (days_since_rebal >= self.rebalance_frequency)
            
            if should_rebalance and idx >= self.min_optimization_history:
                # Get eligible assets (no survivorship bias)
                eligible_assets = self.get_eligible_assets_at_date(current_date)
                
                if len(eligible_assets) >= 5:
                    # Optimize portfolio for current regime
                    target_weights = self.optimize_portfolio(regime, current_date, eligible_assets)
                    
                    # Determine short allocation based on regime
                    target_short_alloc = self.short_allocation if regime == 0 else 0.0
                    
                    # ============================================
                    # STEP 4: EXECUTE TRADES (at next Open)
                    # ============================================
                    # Find next trading day
                    next_idx = idx + 1
                    if next_idx < len(features_df):
                        next_date = features_df.iloc[next_idx]['Date']
                        
                        trade_costs_today = 0.0
                        
                        # ---------------------------------------
                        # CLOSE OLD POSITIONS (sell at Open[T+1])
                        # ---------------------------------------
                        assets_to_close = [asset for asset in positions.keys() 
                                         if asset not in target_weights or target_weights.get(asset, 0) == 0]
                        
                        for asset in assets_to_close:
                            if positions[asset] != 0 and asset in self.asset_data:
                                asset_df = self.asset_data[asset]
                                exec_row = asset_df[asset_df['Date'] == next_date]
                                
                                if not exec_row.empty:
                                    exec_price = exec_row.iloc[0]['Open']
                                    shares = positions[asset]
                                    trade_value = shares * exec_price
                                    
                                    # Calculate costs
                                    costs = self.calculate_trade_costs(trade_value)
                                    trade_costs_today += costs
                                    
                                    # Update cash
                                    cash += trade_value - costs
                                    
                                    # Log trade
                                    trade_history.append({
                                        'date': next_date,
                                        'asset': asset,
                                        'action': 'CLOSE',
                                        'shares': shares,
                                        'price': exec_price,
                                        'value': trade_value,
                                        'cost': costs,
                                        'regime': self.states[regime]
                                    })
                                    
                                    # Clear position
                                    positions[asset] = 0
                        
                        # ---------------------------------------
                        # OPEN/ADJUST LONG POSITIONS (at Open[T+1])
                        # ---------------------------------------
                        for asset, target_weight in target_weights.items():
                            if asset in self.asset_data:
                                asset_df = self.asset_data[asset]
                                exec_row = asset_df[asset_df['Date'] == next_date]
                                
                                if not exec_row.empty:
                                    exec_price = exec_row.iloc[0]['Open']
                                    
                                    # Calculate target value
                                    target_value = portfolio_value * target_weight
                                    
                                    # Current value
                                    current_shares = positions.get(asset, 0)
                                    current_value = current_shares * exec_price
                                    
                                    # Trade needed
                                    trade_value = target_value - current_value
                                    
                                    # Only trade if significant change
                                    if abs(trade_value) > portfolio_value * self.trade_size_floor_pct:
                                        shares_to_trade = trade_value / exec_price
                                        
                                        # Calculate costs
                                        costs = self.calculate_trade_costs(trade_value)
                                        trade_costs_today += costs
                                        
                                        # Update positions
                                        positions[asset] = positions.get(asset, 0) + shares_to_trade
                                        
                                        # Update cash
                                        cash -= (trade_value + costs)
                                        
                                        # Log trade
                                        action = 'BUY' if shares_to_trade > 0 else 'SELL'
                                        trade_history.append({
                                            'date': next_date,
                                            'asset': asset,
                                            'action': action,
                                            'shares': abs(shares_to_trade),
                                            'price': exec_price,
                                            'value': abs(trade_value),
                                            'cost': costs,
                                            'regime': self.states[regime]
                                        })
                        
                        # ---------------------------------------
                        # MANAGE SHORT POSITION (at Open[T+1])
                        # ---------------------------------------
                        if self.short_symbol in self.asset_data:
                            short_df = self.asset_data[self.short_symbol]
                            short_exec_row = short_df[short_df['Date'] == next_date]
                            
                            if not short_exec_row.empty:
                                short_exec_price = short_exec_row.iloc[0]['Open']
                                
                                target_short_value = portfolio_value * target_short_alloc
                                current_short_value = short_position * short_exec_price
                                
                                trade_value_short = target_short_value - current_short_value
                                
                                if abs(trade_value_short) > portfolio_value * self.trade_size_floor_pct:
                                    shares_to_trade_short = trade_value_short / short_exec_price
                                    
                                    costs = self.calculate_trade_costs(trade_value_short)
                                    trade_costs_today += costs
                                    
                                    short_position += shares_to_trade_short
                                    cash -= (trade_value_short + costs)
                                    
                                    action = 'SHORT' if shares_to_trade_short > 0 else 'COVER'
                                    trade_history.append({
                                        'date': next_date,
                                        'asset': self.short_symbol,
                                        'action': action,
                                        'shares': abs(shares_to_trade_short),
                                        'price': short_exec_price,
                                        'value': abs(trade_value_short),
                                        'cost': costs,
                                        'regime': self.states[regime]
                                    })
                        
                        # Update rebalance tracker
                        last_rebalance_idx = idx
        
        # ================================================================
        # PHASE 4: CALCULATE BENCHMARK
        # ================================================================
        if self.verbose:
            print(f"\nPHASE 4: BENCHMARK CALCULATION")
            print("-"*80)
        
        benchmark_data = None
        if self.benchmark_symbol in self.asset_data:
            bench_df = self.asset_data[self.benchmark_symbol].copy()
            bench_df = bench_df[bench_df['Date'].isin(features_df['Date'])]
            
            bench_df['return'] = bench_df['Close'].pct_change()
            
            # Buy and hold with same capital
            first_close = bench_df['Close'].iloc[0]
            shares_bench = self.initial_capital / first_close
            bench_df['value'] = shares_bench * bench_df['Close']
            
            benchmark_data = bench_df
            
            if self.verbose:
                final_bench_value = bench_df['value'].iloc[-1]
                bench_return = (final_bench_value / self.initial_capital - 1) * 100
                print(f"✓ Benchmark ({self.benchmark_symbol}): {bench_return:.2f}%")
        
        # ================================================================
        # PHASE 5: PERFORMANCE METRICS
        # ================================================================
        if self.verbose:
            print(f"\nPHASE 5: PERFORMANCE METRICS")
            print("-"*80)
        
        portfolio_df = pd.DataFrame(portfolio_history)
        regime_df = pd.DataFrame(regime_history)
        
        portfolio_df['return'] = portfolio_df['portfolio_value'].pct_change()
        
        # Calculate metrics
        final_value = portfolio_df['portfolio_value'].iloc[-1]
        total_return = (final_value / self.initial_capital) - 1
        
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        
        returns = portfolio_df['return'].dropna()
        volatility = returns.std() * np.sqrt(252)
        
        sharpe = (annual_return - self.risk_free_rate) / volatility if volatility > 0.001 else 0
        
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        
        # Regime statistics
        regime_stats = regime_df.groupby('regime').size()
        
        metrics = {
            'initial_capital': self.initial_capital,
            'final_value': final_value,
            'total_return': total_return,
            'annual_return': annual_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'num_trades': len(trade_history),
            'regime_distribution': regime_stats.to_dict()
        }
        
        if self.verbose:
            print(f"✓ Final Value: ${final_value:,.0f}")
            print(f"✓ Total Return: {total_return:.2%}")
            print(f"✓ Annual Return: {annual_return:.2%}")
            print(f"✓ Sharpe Ratio: {sharpe:.2f}")
            print(f"✓ Max Drawdown: {max_dd:.2%}")
            print(f"✓ Trades: {len(trade_history)}")
        
        # Store results
        self.backtest_results = {
            'portfolio_history': portfolio_df,
            'regime_history': regime_df,
            'trade_history': trade_history,
            'benchmark_data': benchmark_data,
            'performance_metrics': metrics,
            'features_data': features_df
        }
        
        return self.backtest_results


# ============================================================================
# VISUALIZATION
# ============================================================================

def plot_results(results: Dict, config: Dict):
    """Create comprehensive visualization"""
    
    portfolio_df = results['portfolio_history']
    regime_df = results['regime_history']
    benchmark_df = results['benchmark_data']
    metrics = results['performance_metrics']
    
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    
    # Plot 1: Portfolio Value
    axes[0].plot(portfolio_df['date'], portfolio_df['portfolio_value'], 
                label='Strategy', linewidth=2, color='blue')
    
    if benchmark_df is not None:
        axes[0].plot(benchmark_df['Date'], benchmark_df['value'],
                    label=f"Benchmark ({config['benchmark_symbol']})", 
                    linewidth=2, color='orange', alpha=0.7)
    
    axes[0].set_title('Portfolio Value', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Value ($)', fontsize=11)
    axes[0].legend(loc='upper left')
    axes[0].grid(alpha=0.3)
    
    # Plot 2: Regime Distribution
    regime_colors = {'Bearish': 'red', 'Sideways': 'gray', 'Bullish': 'green'}
    
    for regime in ['Bearish', 'Sideways', 'Bullish']:
        mask = regime_df['regime'] == regime
        if mask.sum() > 0:
            axes[1].scatter(regime_df.loc[mask, 'date'], 
                          [regime]*mask.sum(),
                          c=regime_colors[regime], 
                          alpha=0.6, s=10)
    
    axes[1].set_title('Regime Detection', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Regime', fontsize=11)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['Bearish', 'Sideways', 'Bullish'])
    axes[1].grid(alpha=0.3)
    
    # Plot 3: Returns Distribution
    returns = portfolio_df['return'].dropna()
    axes[2].hist(returns * 100, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[2].set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Return (%)', fontsize=11)
    axes[2].set_ylabel('Frequency', fontsize=11)
    axes[2].grid(alpha=0.3)
    
    # Plot 4: Drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max * 100
    
    axes[3].fill_between(range(len(drawdown)), drawdown, 0, 
                         alpha=0.3, color='red')
    axes[3].plot(drawdown, color='darkred', linewidth=1.5)
    axes[3].set_title('Strategy Drawdown', fontsize=14, fontweight='bold')
    axes[3].set_xlabel('Days', fontsize=11)
    axes[3].set_ylabel('Drawdown (%)', fontsize=11)
    axes[3].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*80)
    print("PERFORMANCE SUMMARY")
    print("="*80)
    print(f"Initial Capital:  ${metrics['initial_capital']:,.0f}")
    print(f"Final Value:      ${metrics['final_value']:,.0f}")
    print(f"Total Return:     {metrics['total_return']:.2%}")
    print(f"Annual Return:    {metrics['annual_return']:.2%}")
    print(f"Volatility:       {metrics['volatility']:.2%}")
    print(f"Sharpe Ratio:     {metrics['sharpe_ratio']:.2f}")
    print(f"Max Drawdown:     {metrics['max_drawdown']:.2%}")
    print(f"Number of Trades: {metrics['num_trades']}")
    
    print(f"\nRegime Distribution:")
    for regime, count in metrics['regime_distribution'].items():
        pct = count / len(regime_df) * 100
        print(f"  {regime}: {count} days ({pct:.1f}%)")
    print("="*80)


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution"""
    
    config = {
        'n_particles': 1000,
        'initial_capital': 100000.0,
        'rebalance_frequency': 21,
        'benchmark_symbol': 'URTH',
        'risk_free_rate': 0.02,
        'commission_rate': 0.0005,
        'bid_ask_spread': 0.0003,
        'market_impact': 0.0002,
        'min_commission': 1.0,
        'max_position_size': 0.15,
        'min_optimization_history': 126,
        'cdar_threshold': -0.05,
        'upside_threshold': 0.10,
        'short_allocation': 0.15,
        'min_asset_success_rate': 0.75,
        'min_data_points_for_optimization': 60,
        'trade_size_floor_pct': 0.005,
        'max_missing_data_pct': 0.20,
        'verbose': True
    }
    
    print("="*80)
    print("PARTICLE FILTER STRATEGY - COMPLETE IMPLEMENTATION")
    print("="*80)
    
    # Initialize and run
    model = SP500RegimeFilter(**config)
    
    results = model.run_backtest(
        start_date="2013-01-01",
        end_date="2018-12-31"
    )
    
    # Visualize results
    print("\n" + "="*80)
    print("GENERATING VISUALIZATIONS")
    print("="*80)
    
    plot_results(results, config)
    
    return results, model


if __name__ == "__main__":
    results, model = main()